# 02 · 20261029 Prompt 工程与链式编排

> 课堂练习：`20261029/` 下 11 个脚本（test1 ~ test11）。
> 本笔记把散落的脚本归纳成一条主线，并把脚本里硬编码的 Key / 局域网地址改成读 `week23/.env`。

## 今日主线

**Prompt 模板** → **少样本 / 动态选例** → **LCEL 组合** → **结构化输出** → **对话记忆**

## 文件 ↔ 主题对照

| 脚本 | 主题 | 关键 API |
|---|---|---|
| test1 | 字符串模板 | `PromptTemplate`、`PromptValue` |
| test2 | 角色化模板 | `ChatPromptTemplate.from_messages` |
| test3 | 历史消息占位 | `MessagesPlaceholder` |
| test4 | Few-shot 三种写法 | `FewShotPromptTemplate`、`FewShotChatMessagePromptTemplate` |
| test5 | Embedding 模型对比 | `OllamaEmbeddings` + 余弦相似度 |
| test7 | 语义动态选例 | `SemanticSimilarityExampleSelector` + FAISS |
| test8 | LCEL 管道 | `prompt \| model` |
| test9 | 输出解析 | `JsonOutputParser`、`PydanticOutputParser` |
| test10 | 内存记忆 | `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` |
| test11 | Redis 记忆 | `RedisChatMessageHistory` |

## 0 · 统一配置

原脚本每个文件都重复写了 `base_url` / `api_key`，还夹着课堂局域网的地址（Ollama `192.168.1.105:11434`、Redis `192.168.2.197:6379`）。

这里统一走 `20261027/lab_config.py`：

- 模型来源由 `.env` 的 `LLM_PROVIDER` 决定（`qwen` / `openai` / `ollama`）
- Key 只存在 `.env`（已 gitignore），不进代码、不进 Git

下面这格只读配置，**不联网、不计费**。

In [1]:
from pathlib import Path
import sys


def find_lab_dir(start: Path) -> Path:
    """向上找 20261027/lab_config.py，复用 week23 的统一配置。"""
    for base in [start, *start.parents]:
        candidate = base / "20261027"
        if (candidate / "lab_config.py").exists():
            return candidate
    raise FileNotFoundError("没找到 20261027/lab_config.py，请在 week23 目录下打开本笔记本")


sys.path.insert(0, str(find_lab_dir(Path.cwd())))

from lab_config import get_model, show_config

show_config()

provider = qwen
model    = qwen3.8-flash
base_url = https://cn-beijing.maas.qianwenaiapi.com/compatible-mode/v1
api_key  = sk-ws-***ef-t
max_tokens = 1024
thinking   = False（true 会白烧 reasoning token）


In [7]:
# 会真正调用模型的单元格都套了这个开关：默认 False，避免误执行烧 token
RUN_DEMOS = False

## 1 · PromptTemplate：字符串模板

`template` 里用 `{变量名}` 占位，`input_variables` 声明有哪些变量。

- `format(...)` 返回 **str**
- `invoke({...})` 返回 **PromptValue**，可再 `.to_messages()` 转成消息列表

这一格完全离线，可以放心执行。

In [2]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="你好，我的名字是{name}，我今年{age}岁，我擅长{skill}！",
    input_variables=["name", "age", "skill"],
)

text = prompt.format(name="World", age=18, skill="编程")
print("format()      ->", type(text).__name__, "|", text)

value = prompt.invoke({"name": "World", "age": 18, "skill": "编程"})
print("invoke()      ->", type(value).__name__)      # PromptValue
print("to_messages() ->", value.to_messages())      # [HumanMessage]

format()      -> str | 你好，我的名字是World，我今年18岁，我擅长编程！
invoke()      -> StringPromptValue
to_messages() -> [HumanMessage(content='你好，我的名字是World，我今年18岁，我擅长编程！', additional_kwargs={}, response_metadata={})]


## 2 · ChatPromptTemplate：按角色写模板

多轮对话要有 role 概念，`from_messages` 用元组 `("system" / "human" / "ai", 内容)` 声明。

同一个模板有四种取值方式，返回类型不同（下面格子直接打印出来对比）：

| 方法 | 返回类型 | 什么时候用 |
|---|---|---|
| `format` | `str` | 只想看拼好的文本 |
| `format_prompt` | `ChatPromptValue` | 需要 PromptValue 对象 |
| `format_messages` | `list[BaseMessage]` | 直接拿去调试或喂给 model |
| `invoke` | `PromptValue` | LCEL 链里的统一入口 |

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是物理{system_name}老师。"),
    ("human", "解释一下什么是量子计算，你好{name}？"),
])

print("format          ->", type(chat_prompt.format(system_name="高级", name="张三")).__name__)
print("format_prompt   ->", type(chat_prompt.format_prompt(system_name="高级", name="张三")).__name__)
print("format_messages ->", type(chat_prompt.format_messages(system_name="高级", name="张三")).__name__)
print("invoke          ->", type(chat_prompt.invoke({"system_name": "高级", "name": "张三"})).__name__)

for m in chat_prompt.format_messages(system_name="高级", name="张三"):
    print(f"  [{m.type:6s}] {m.content}")

format          -> str
format_prompt   -> ChatPromptValue
format_messages -> list
invoke          -> ChatPromptValue
  [system] 你是物理高级老师。
  [human ] 解释一下什么是量子计算，你好张三？


## 3 · MessagesPlaceholder：给历史消息留位置

多轮对话的历史长度不固定，模板里先放一个占位符，调用时把历史列表传进去。

这是后面第 7 节「记忆」能自动注入历史的前提。

In [4]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是物理{system_name}老师。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

messages = chat_prompt.format_messages(
    system_name="高级",
    question="他有什么特征？",
    history=[
        ("human", "解释一下什么是量子计算？"),
        ("assistant", "量子计算利用量子叠加与纠缠并行处理信息。"),
    ],
)

for m in messages:
    print(f"[{m.type:9s}] {m.content}")

[system   ] 你是物理高级老师。
[human    ] 解释一下什么是量子计算？
[ai       ] 量子计算利用量子叠加与纠缠并行处理信息。
[human    ] 他有什么特征？


## 4 · Few-shot：用示例约束输出风格

三种写法（test4.py 都试了一遍）：

| 写法 | 说明 | 特点 |
|---|---|---|
| 手写进 system | `"……样例：{example}"` | 最省事，示例多了 prompt 会很乱 |
| `FewShotPromptTemplate` | `prefix` + `example_prompt` + `suffix` | 纯文本拼接，输出 `str` |
| `FewShotChatMessagePromptTemplate` | 示例本身就是 human/ai 消息对 | **可嵌进 `ChatPromptTemplate`**，推荐 |

下面演示第三种：系统提示 + 3 组示例 + 真实问题，最终拼成一串消息。

In [5]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"question": "你好吗?", "answer": "师哥，我很好"},
    {"question": "今天周几?", "answer": "师哥，今天星期二"},
    {"question": "天气好吗?", "answer": "师哥，是的，今天天气很好"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}"),
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个中二的服务员，请用类似的风格回答问题。"),
    few_shot,
    ("human", "{input}"),
])

print(final_prompt.invoke({"input": "你好吗?"}))

messages=[SystemMessage(content='你是一个中二的服务员，请用类似的风格回答问题。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好吗?', additional_kwargs={}, response_metadata={}), AIMessage(content='师哥，我很好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='今天周几?', additional_kwargs={}, response_metadata={}), AIMessage(content='师哥，今天星期二', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='天气好吗?', additional_kwargs={}, response_metadata={}), AIMessage(content='师哥，是的，今天天气很好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你好吗?', additional_kwargs={}, response_metadata={})]


## 5 · 按需选例：语义相似度 + 向量库

示例一多，全都塞进 prompt 既费 token 又互相干扰。`SemanticSimilarityExampleSelector` 先把示例写进向量库，再按「问题」的相似度取 top-k。

**test5.py 做的是 embedding 模型选型实验**（`qwen3-embedding:0.6b` vs `bge-large`）：

- 它算的是「问题向量 · 自己答案向量」余弦相似度的均值 → 实际衡量的是 **QA 语义对齐度**，不是检索效果
- 更能反映选型效果的是：同义问法之间相似度高 + 不相关问题相似度低
- 原脚本每个样例调了 6 次 `embed_query`，建议先批量 embed 再算，否则 10 个样例就是 120 次请求

⚠️ 这一段需要 embedding 服务（Ollama）和 `faiss-cpu`（`uv add faiss-cpu`），所以放在 `RUN_DEMOS` 开关后面。

In [8]:
if RUN_DEMOS:
    from langchain_core.example_selectors import SemanticSimilarityExampleSelector
    from langchain_core.prompts import FewShotPromptTemplate
    from langchain_community.vectorstores import FAISS
    from langchain_ollama import OllamaEmbeddings

    embedding = OllamaEmbeddings(model="bge-large", base_url="http://127.0.0.1:11434")

    selector = SemanticSimilarityExampleSelector.from_examples(
        examples=examples,          # 复用上一格的 examples
        embeddings=embedding,
        vectorstore_cls=FAISS,
        k=2,                        # 只取最相似的 2 条
    )

    semantic_prompt = FewShotPromptTemplate(
        example_selector=selector,
        example_prompt=PromptTemplate(
            input_variables=["question", "answer"],
            template="问题: {question}\n回答: {answer}\n",
        ),
        prefix="基于相似问题，请回答：\n",
        suffix="问题: {question}\n回答:",
        input_variables=["question"],
    )

    print(semantic_prompt.invoke({"question": "什么是卷积神经网络"}))
else:
    print("（RUN_DEMOS=False，跳过。需要 Ollama embedding 服务 + faiss-cpu）")

（RUN_DEMOS=False，跳过。需要 Ollama embedding 服务 + faiss-cpu）


## 6 · LCEL：`prompt | model | parser`

`|` 把组件串成链，前一个的输出直接喂给后一个。

结构化输出两条路：

| 解析器 | 结果 | 特点 |
|---|---|---|
| `JsonOutputParser` | `dict` | 轻量，字段完全靠 prompt 约定 |
| `PydanticOutputParser` | Pydantic 对象 | 有类型校验，字段 `description` 会自动写进格式说明 |

Pydantic 版本先用 `get_format_instructions()` 生成「输出格式说明」，再用 `partial` 提前绑定进模板——原脚本把它写死进 `template` 却没声明进 `input_variables`，容易踩坑。

In [9]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field


class OrderInfo(BaseModel):
    order_id: str = Field(description="订单号")
    product_name: str = Field(description="商品名称")
    amount: float = Field(description="金额")
    status: str = Field(description="状态")


parser = PydanticOutputParser(pydantic_object=OrderInfo)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"order_id": {"description": "订单号", "title": "Order Id", "type": "string"}, "product_name": {"description": "商品名称", "title": "Product Name", "type": "string"}, "amount": {"description": "金额", "title": "Amount", "type": "number"}, "status": {"description": "状态", "title": "Status", "type": "string"}}, "required": ["order_id", "product_name", "amount", "status"]}
```


In [10]:
base_prompt = PromptTemplate(
    input_variables=["text"],
    template="你是一个数据提取助手，请从以下{text}中提取内容，并按指定格式返回。{format_instructions}",
)

# partial 先把 format_instructions 填死，invoke 时只需要传 text
prompt = base_prompt.partial(format_instructions=parser.get_format_instructions())

if RUN_DEMOS:
    chain = prompt | get_model(max_tokens=512) | parser     # 模板 → 模型 → 解析器
    info = chain.invoke(
        {"text": "用户：收纳箱单号44002211，仓库拣货了吗？\n客服：收纳箱45.80元，目前待发货，正在拣货。"}
    )
    print(type(info).__name__, info)
    print(info.order_id, info.product_name, info.amount, info.status)
    print(info.model_dump())      # Pydantic 对象 → dict
else:
    print("（RUN_DEMOS=False，只预览最终 prompt）\n")
    print(prompt.format(text="用户：收纳箱单号44002211，仓库拣货了吗？……"))

（RUN_DEMOS=False，只预览最终 prompt）

你是一个数据提取助手，请从以下用户：收纳箱单号44002211，仓库拣货了吗？……中提取内容，并按指定格式返回。The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"order_id": {"description": "订单号", "title": "Order Id", "type": "string"}, "product_name": {"description": "商品名称", "title": "Product Name", "type": "string"}, "amount": {"description": "金额", "title": "Amount", "type": "number"}, "status": {"description": "状态", "title": "Status", "type": "string"}}, "required": ["order_id", "product_name", "amount", "status"]}
```


## 7 · 对话记忆：RunnableWithMessageHistory

三步走：

1. 准备 `get_session_history(session_id)` → 返回一个历史容器
2. 模板里用 `MessagesPlaceholder(variable_name="history")` 留位置
3. 用 `RunnableWithMessageHistory` 包装链，每次 `invoke` 传 `config={"configurable": {"session_id": ...}}`

`session_id` 不同 → 历史互相隔离，天然支持多用户。

**换存储后端只换 `get_session_history` 一个函数**：

| 后端 | 类 | 场景 |
|---|---|---|
| 内存 dict | `InMemoryChatMessageHistory` | 练习、单进程，重启即丢 |
| Redis | `RedisChatMessageHistory` | 多进程 / 服务重启不丢 |
| 数据库 | `SQLChatMessageHistory` | 需要持久化与查询 |

In [11]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """同一个 session_id 共用一段历史；不同 id 完全隔离。"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


mem_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的 AI 助手"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

In [12]:
if RUN_DEMOS:
    chain_with_history = RunnableWithMessageHistory(
        mem_prompt | get_model(max_tokens=256),
        get_session_history,
        input_messages_key="input",        # 用户输入放在哪个键
        history_messages_key="history",    # 历史填进哪个占位符
    )
    config = {"configurable": {"session_id": "user-001"}}

    r1 = chain_with_history.invoke({"input": "你好，我的名字是张三，我是你的老板"}, config=config)
    print("第 1 轮:", r1.content)

    r2 = chain_with_history.invoke({"input": "我叫什么？"}, config=config)
    print("第 2 轮:", r2.content)

    print("user-001 历史条数:", len(store["user-001"].messages))

### Redis 版：链完全不用改

`RunnableWithMessageHistory` 的参数一个字都不用动，只把第 1 步换成 `RedisChatMessageHistory`。

In [13]:
import os


def get_redis_history(session_id: str):
    """换成 Redis 只需改这一个函数，链的定义保持不变。"""
    from langchain_community.chat_message_histories import RedisChatMessageHistory

    url = os.getenv("REDIS_URL", "redis://127.0.0.1:6379/13")
    return RedisChatMessageHistory(session_id, url=url)


# 用法完全一致：
# chain_with_history = RunnableWithMessageHistory(
#     mem_prompt | get_model(max_tokens=256),
#     get_redis_history,
#     input_messages_key="input",
#     history_messages_key="history",
# )
print("get_redis_history 已定义（真正连接发生在 invoke 时）")

get_redis_history 已定义（真正连接发生在 invoke 时）


## 8 · 今日踩坑与改进清单

| # | 问题 | 建议 |
|---|---|---|
| 1 | 每个脚本都硬编码 `api_key` | 改读 `.env`（本笔记已做）；**不要提交到 GitHub** |
| 2 | 课堂局域网 IP 写死在代码里 | 换成配置项（如 `REDIS_URL`），否则换机器跑不通 |
| 3 | test5 的指标是「问题 vs 自己的答案」 | 想评 embedding 质量，改成同义 / 异义问题对对比 |
| 4 | test5 每个样例重复 `embed_query` 6 次 | 先批量 embed 一次，再算相似度矩阵 |
| 5 | test9 `template` 用了 `{format_instructions}` 但没声明进 `input_variables` | 用 `prompt.partial(...)` 绑定 |
| 6 | FAISS 需要额外依赖 | `uv add faiss-cpu` |
| 7 | `qwen3.8-*` 默认开思考，`max_tokens` 小的话额度会被 reasoning 吃掉 | `.env` 设 `QWEN_ENABLE_THINKING=false` |
| 8 | test4 `load_model()` 的 `provider` 参数没被用到 | 删掉该参数，或真正传给 `model_provider` |
| 9 | test1 / test3 里有未使用的 `model`、大片注释掉的试验代码 | 整理进笔记，保持脚本可运行 |

## 小结

今天把 LangChain 的「提示词前半段」走完了：

```mermaid
graph LR
    A[PromptTemplate] --> B[ChatPromptTemplate]
    B --> C[MessagesPlaceholder]
    C --> D[Few-shot 示例]
    D --> E[语义选例]
    E --> F[LCEL 链]
    F --> G[OutputParser]
    F --> H[MessageHistory]
```

**下一步**：文档加载与切分（`langchain-text-splitters`）→ 向量库（Chroma）→ **RAG 检索链**。